# ECMWF AIFS-single Analysis — TGPY (Maurice Bishop International Airport, Grenada)

**Station:** TGPY — Maurice Bishop International Airport  
**Location:** 12.0042°N, 61.7862°W | **Timezone:** UTC-4

---

## Data layout

```
data/2026-06-01_12Z/
│
├── Verification snapshots (Sections 3–4)
│   ├── msl_f024.grib2
│   ├── sfc_f024.grib2
│   ├── pl_f024.grib2
│   └── wave_f024.grib2
│
└── Forecast groups (Sections 5–6)
    ├── sfc_f000-f120_3h.grib2       surface (2t, 10u/v, 10fg, 2d, tcc, tp, cp, ro, tcwv, msl, sp, cape, blh)
    ├── pl_f000-f120_12h.grib2       upper air (gh, t, u, v, w, q, r, vo, d — 10 levels)
    ├── wave_f000-f120_3h.grib2      ocean waves (swh, mwd, mwp, pp1d)
    ├── ens_prob_precip_d1-d5.grib2  ENS precip probability (tpg1, tpg5)
    └── ens_prob_wave_d1-d5.grib2    ENS wave probability (swhg2, swhg3, swhg4)
```

---

## Sections

1. Setup + shared utilities
2. Nearest-point verification
3. Surface snapshot at T+24h
4. Upper-air profile at T+24h
5. Surface time series (0–120h, 6-hourly)
6. Step-by-step data — all variables grouped by forecast hour

In [ ]:
import xarray as xr
import cfgrib
import numpy as np
from pathlib import Path

# ── Set this to the run you want to analyse ──────────────────────────────────
# Must match a folder under data/ created by ecmwf-opendata.ipynb.
# Format: YYYY-MM-DD_HHZ  (HH = 00 or 12)
RUN_DIR = Path("data/2026-06-01_12Z")
# ─────────────────────────────────────────────────────────────────────────────

# TGPY — Maurice Bishop International Airport, Grenada
TGPY_LAT =  12.0042
TGPY_LON = -61.7862

print(f"Run directory : {RUN_DIR.resolve()}")
print(f"Exists        : {RUN_DIR.exists()}")
print(f"Station       : TGPY  ({TGPY_LAT}°N, {TGPY_LON}°W)")

In [ ]:
import pandas as pd

UTC_OFFSET = pd.Timedelta(hours=-4)   # AST — Grenada does not observe DST

def nearest_point(ds):
    """Select the grid point nearest to TGPY."""
    return ds.sel(latitude=TGPY_LAT, longitude=TGPY_LON, method="nearest", tolerance=0.25)

def deg_to_cardinal(deg):
    dirs = ["N","NNE","NE","ENE","E","ESE","SE","SSE",
            "S","SSW","SW","WSW","W","WNW","NW","NNW"]
    return dirs[round(deg / 22.5) % 16]

def grib_to_df(filepath):
    """Load a multi-step GRIB file at TGPY into a DataFrame indexed by valid time."""
    datasets = cfgrib.open_datasets(str(filepath))
    series = {}
    for ds in datasets:
        pt = nearest_point(ds)
        for var in pt.data_vars:
            da  = pt[var]
            vt  = da.valid_time.values
            idx = pd.to_datetime([vt]) if vt.ndim == 0 else pd.to_datetime(vt)
            val = [float(da.values)] if vt.ndim == 0 else da.values.astype(float)
            series[var] = pd.Series(val, index=idx)
    return pd.DataFrame(series).sort_index()

## Section 2 — Nearest-point verification

ECMWF data is on a 0.25° global grid (~28 km spacing). There is no grid point exactly at TGPY, so we use `xarray`'s `.sel(method="nearest")` to find the closest one. This cell confirms which grid point is selected and how far it is from the actual station.

In [ ]:
ds_check = xr.open_dataset(str(RUN_DIR / "msl_f024.grib2"), engine="cfgrib")
pt = nearest_point(ds_check)

actual_lat = float(pt.latitude)
actual_lon = float(pt.longitude)
dist_km    = np.sqrt((actual_lat - TGPY_LAT)**2 + (actual_lon - TGPY_LON)**2) * 111

print(f"Requested : {TGPY_LAT:.4f}°N, {TGPY_LON:.4f}°W")
print(f"Nearest   : {actual_lat:.4f}°N, {actual_lon:.4f}°W")
print(f"Offset    : ~{dist_km:.1f} km")

## Section 3 — Surface snapshot at T+24h

Single-step surface conditions at the TGPY grid point.

In [ ]:
sfc_datasets = cfgrib.open_datasets(str(RUN_DIR / "sfc_f024.grib2"))

sfc = {}
for ds in sfc_datasets:
    for var in ds.data_vars:
        sfc[var] = nearest_point(ds)

print("Surface variables:", list(sfc.keys()))
print()

t2m_k  = float(sfc["t2m"]["t2m"])
t2m_c  = t2m_k - 273.15
print(f"2m Temperature : {t2m_c:.1f} °C")

msl_hpa = float(sfc["msl"]["msl"]) / 100
print(f"MSL Pressure   : {msl_hpa:.1f} hPa")

u10   = float(sfc["u10"]["u10"])
v10   = float(sfc["v10"]["v10"])
wspd  = np.sqrt(u10**2 + v10**2)
wdir  = (270 - np.degrees(np.arctan2(v10, u10))) % 360
print(f"10m Wind Speed : {wspd:.1f} m/s  ({wspd * 1.944:.1f} kt)")
print(f"10m Wind Dir   : {wdir:.0f}°")
print(f"\nValid time     : {sfc['msl'].valid_time.values}")

## Section 4 — Upper-air profile at T+24h

Vertical profile at the TGPY grid point — equivalent to a model sounding.

In [ ]:
ua_datasets = cfgrib.open_datasets(str(RUN_DIR / "pl_f024.grib2"))
ua = nearest_point(ua_datasets[0])

levels = ua.coords["isobaricInhPa"].values

print(f"Upper-air profile at TGPY — valid {ua.valid_time.values}")
print()
print(f"{'Level':>8}  {'Temp':>8}  {'GH':>7}  {'U':>7}  {'V':>7}  {'Speed':>8}  {'Dir':>6}")
print(f"{'(hPa)':>8}  {'(°C)':>8}  {'(m)':>7}  {'(m/s)':>7}  {'(m/s)':>7}  {'(m/s)':>8}  {'(°)':>6}")
print("-" * 68)

for lev in levels:
    pt   = ua.sel(isobaricInhPa=lev)
    t_c  = float(pt["t"]) - 273.15
    gh   = float(pt["gh"])
    u    = float(pt["u"])
    v    = float(pt["v"])
    spd  = np.sqrt(u**2 + v**2)
    wdir = (270 - np.degrees(np.arctan2(v, u))) % 360
    print(f"{lev:>8.0f}  {t_c:>8.1f}  {gh:>7.0f}  {u:>7.1f}  {v:>7.1f}  {spd:>8.1f}  {wdir:>6.0f}")

## Section 5 — Surface time series (0–120h, 6-hourly)

Overview of surface conditions across the 5-day forecast window.

In [ ]:
# Section 5 reads from the Group 1 batch file.
# Filter to every other step (6-hourly) for a concise overview — Section 6 has full 3-hourly detail.

df_s5 = grib_to_df(RUN_DIR / "sfc_f000-f120_3h.grib2")
df_s5 = df_s5.iloc[::2]   # T+0, T+6, T+12, ..., T+120 — 21 rows

rows = []
for ts, row in df_s5.iterrows():
    step_h = int((pd.Timestamp(ts) - pd.Timestamp(df_s5.index[0])).total_seconds() / 3600)
    t2m_c  = float(row["t2m"]) - 273.15
    msl_h  = float(row["msl"]) / 100
    u10    = float(row["u10"])
    v10    = float(row["v10"])
    wspd   = np.sqrt(u10**2 + v10**2)
    wdir   = (270 - np.degrees(np.arctan2(v10, u10))) % 360
    rows.append({
        "step": step_h, "valid_time": ts,
        "t2m_c": t2m_c, "msl_hpa": msl_h,
        "wspd_kt": wspd * 1.944, "wdir": wdir,
    })

print(f"{'Step':>5}  {'Valid time (UTC)':>20}  {'T2m':>7}  {'MSL':>8}  {'Spd (kt)':>9}  {'Dir':>5}")
print("-" * 60)
for r in rows:
    vt = str(r["valid_time"])[:16].replace("T", " ")
    print(f"{r['step']:>5}  {vt:>20}  {r['t2m_c']:>7.1f}  {r['msl_hpa']:>8.1f}  {r['wspd_kt']:>9.1f}  {r['wdir']:>5.0f}")

## Section 6 — Step-by-step data

All available variables grouped by forecast hour. Coverage: T+0 to T+120h.

- **Surface** (3-hourly) — temperature, dewpoint, pressure, wind, actual gust, cloud cover, precipitation, runoff, TCWV, MUCAPE
- **Wave** (3-hourly) — SWH, direction, mean period, peak period
- **Upper air** (12-hourly) — full vertical profile at 10 levels (200–1000 hPa): temperature, geopotential height, wind, humidity, vorticity, divergence, vertical velocity

This is the raw model data at TGPY. Apply your own thresholds and summaries from here.

In [ ]:
# ── Load surface and wave ─────────────────────────────────────────────────────
df_sfc  = grib_to_df(RUN_DIR / "sfc_f000-f120_3h.grib2")
df_wave = grib_to_df(RUN_DIR / "wave_f000-f120_3h.grib2")

# ── Load upper air ────────────────────────────────────────────────────────────
ua_vars = {}
ua_lev  = None
ua_vt   = None

for ds in cfgrib.open_datasets(str(RUN_DIR / "pl_f000-f120_12h.grib2")):
    pt = nearest_point(ds)
    if "isobaricInhPa" not in pt.dims:
        continue
    for var in pt.data_vars:
        ua_vars[var] = pt[var]
    if ua_lev is None:
        ua_lev = sorted(pt.coords["isobaricInhPa"].values.tolist(), reverse=True)
    if ua_vt is None:
        vt = pt.valid_time.values
        ua_vt = pd.to_datetime(vt if vt.ndim > 0 else [vt])

# AIFS uses "tcw" (total column water) instead of "tcwv"
if "tcw" in df_sfc.columns and "tcwv" not in df_sfc.columns:
    df_sfc = df_sfc.rename(columns={"tcw": "tcwv"})

# ── Core derived surface columns ──────────────────────────────────────────────
df_sfc["t2m_c"]   = df_sfc["t2m"] - 273.15
df_sfc["wspd_kt"] = np.sqrt(df_sfc["u10"]**2 + df_sfc["v10"]**2) * 1.944
df_sfc["wdir"]    = (270 - np.degrees(np.arctan2(df_sfc["v10"], df_sfc["u10"]))) % 360
df_sfc["msl_hpa"] = df_sfc["msl"] / 100
df_sfc["tp_mm"]   = df_sfc["tp"].diff().clip(lower=0) * 1000

if "fg10" in df_sfc.columns:
    df_sfc["fg10_kt"] = df_sfc["fg10"] * 1.944
else:
    df_sfc["fg10_kt"] = df_sfc["wspd_kt"] * 1.4

if "d2m" in df_sfc.columns:
    df_sfc["d2m_c"] = df_sfc["d2m"] - 273.15
if "tcc" in df_sfc.columns:
    df_sfc["tcc_pct"] = df_sfc["tcc"] * 100
if "sp" in df_sfc.columns:
    df_sfc["sp_hpa"] = df_sfc["sp"] / 100
if "ro" in df_sfc.columns:
    df_sfc["ro_mm"] = df_sfc["ro"].diff().clip(lower=0) * 1000

# ── Extended tropical derived columns ─────────────────────────────────────────
period_s = (pd.to_datetime(df_sfc.index[1]) - pd.to_datetime(df_sfc.index[0])).total_seconds()

if "skt"  in df_sfc.columns: df_sfc["skt_c"]      = df_sfc["skt"] - 273.15
if "sst"  in df_sfc.columns: df_sfc["sst_c"]       = df_sfc["sst"] - 273.15
if "lcc"  in df_sfc.columns: df_sfc["lcc_pct"]     = df_sfc["lcc"] * 100
if "mcc"  in df_sfc.columns: df_sfc["mcc_pct"]     = df_sfc["mcc"] * 100
if "hcc"  in df_sfc.columns: df_sfc["hcc_pct"]     = df_sfc["hcc"] * 100
if "cp"   in df_sfc.columns: df_sfc["cp_mm"]        = df_sfc["cp"].diff().clip(lower=0) * 1000
if "u100" in df_sfc.columns and "v100" in df_sfc.columns:
    df_sfc["wspd100_kt"] = np.sqrt(df_sfc["u100"]**2 + df_sfc["v100"]**2) * 1.944
if "ssrd" in df_sfc.columns: df_sfc["ssrd_wm2"]    = df_sfc["ssrd"].diff().clip(lower=0) / period_s
if "strd" in df_sfc.columns: df_sfc["strd_wm2"]    = df_sfc["strd"].diff().clip(lower=0) / period_s
# AIFS has no ttr → olr_wm2 will be NaN; no blh, mucin, mucape
df_sfc["olr_wm2"] = float("nan")
df_sfc["blh_m"]   = float("nan")
df_sfc["cin"]     = float("nan")
df_sfc["cape"]    = float("nan")
df_sfc["lhf"]     = float("nan")
df_sfc["shf"]     = float("nan")

# AIFS mucape not available via open-data — leave as NaN
if "mucape" in df_sfc.columns:
    df_sfc["cape"] = df_sfc["mucape"]

df_wave["swh_ft"] = df_wave["swh"] * 3.281

# ── Column headers ────────────────────────────────────────────────────────────
run_init = df_sfc.index[0]

def step_cols(index):
    cols = []
    for ts in pd.to_datetime(index):
        step_h = int((ts - run_init).total_seconds() / 3600)
        lt     = (ts + UTC_OFFSET).strftime("%d%b %H:%M")
        cols.append(f"T+{step_h:03d}\n{lt}")
    return cols

sfc_cols = step_cols(df_sfc.index)
wav_cols = step_cols(df_wave.index)
ua_cols  = step_cols(ua_vt)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 12)

print(f"Surface  : {len(df_sfc)} steps  |  cols: {list(df_sfc.columns)}")
print(f"Wave     : {len(df_wave)} steps")
print(f"Upper air: {len(ua_vt)} steps  |  levels: {ua_lev}")
print(f"UA vars  : {list(ua_vars.keys())}")

In [ ]:
# ── Surface table — rows = variables, columns = time steps ───────────────────
sfc_table = pd.DataFrame(
    {
        "T2m (°C)":         df_sfc["t2m_c"].round(1).values,
        "Dewpoint (°C)":    df_sfc["d2m_c"].round(1).values         if "d2m_c"   in df_sfc.columns else ["—"] * len(df_sfc),
        "MSL (hPa)":        df_sfc["msl_hpa"].round(1).values,
        "SP (hPa)":         df_sfc["sp_hpa"].round(1).values,
        "Wind dir":         [deg_to_cardinal(d) for d in df_sfc["wdir"]],
        "Wind (kt)":        df_sfc["wspd_kt"].round(1).values,
        "Gust (kt)":        df_sfc["fg10_kt"].fillna(0).round(0).astype(int).values,
        "Cloud (%)":        df_sfc["tcc_pct"].fillna(0).round(0).astype(int).values  if "tcc_pct" in df_sfc.columns else ["—"] * len(df_sfc),
        "Precip 3h (mm)":   df_sfc["tp_mm"].fillna(0).round(2).values,
        "Runoff 3h (mm)":   df_sfc["ro_mm"].fillna(0).round(2).values,
        "TCWV (kg/m²)":     df_sfc["tcwv"].round(1).values,
        "MUCAPE (J/kg)":    df_sfc["mucape"].fillna(0).round(0).astype(int).values   if "mucape"  in df_sfc.columns else ["—"] * len(df_sfc),
    },
    index=sfc_cols,
).T

print("SURFACE")
print("=" * 60)
print(sfc_table.to_string())

# ── Wave table ────────────────────────────────────────────────────────────────
wav_table = pd.DataFrame(
    {
        "SWH (m)":          df_wave["swh"].round(2).values,
        "SWH (ft)":         df_wave["swh_ft"].round(1).values,
        "Dir":              [deg_to_cardinal(d) for d in df_wave["mwd"]],
        "Mean period (s)":  df_wave["mwp"].round(1).values,
        "Peak period (s)":  df_wave["pp1d"].round(1).values if "pp1d" in df_wave.columns else ["—"] * len(df_wave),
    },
    index=wav_cols,
).T

print()
print("WAVE")
print("=" * 60)
print(wav_table.to_string())

# ── Upper-air tables — one per variable, rows = levels, columns = 12h steps ──
# ua_vars is a dict: shortName → DataArray(step, isobaricInhPa)

def ua_table(shortname, transform=lambda x: x, decimals=1):
    rows = {}
    for lev in ua_lev:
        da   = ua_vars[shortname].sel(isobaricInhPa=lev)
        vals = da.values if da.values.ndim > 0 else [float(da.values)]
        rows[f"{lev:.0f} hPa"] = [round(transform(v), decimals) for v in vals]
    return pd.DataFrame(rows, index=ua_cols).T

# Wind speed and direction derived from u/v
ws_rows, wd_rows = {}, {}
for lev in ua_lev:
    u_vals = ua_vars["u"].sel(isobaricInhPa=lev).values
    v_vals = ua_vars["v"].sel(isobaricInhPa=lev).values
    if u_vals.ndim == 0:
        u_vals, v_vals = [float(u_vals)], [float(v_vals)]
    ws_rows[f"{lev:.0f} hPa"] = [round(np.sqrt(u**2 + v**2) * 1.944, 1) for u, v in zip(u_vals, v_vals)]
    wd_rows[f"{lev:.0f} hPa"] = [deg_to_cardinal((270 - np.degrees(np.arctan2(v, u))) % 360) for u, v in zip(u_vals, v_vals)]

ua_tables = {
    "Temperature (°C)":        ua_table("t",  lambda x: x - 273.15, 1),
    "Geopotential ht (m)":     ua_table("gh", lambda x: x,          0),
    "Wind speed (kt)":         pd.DataFrame(ws_rows, index=ua_cols).T,
    "Wind direction":          pd.DataFrame(wd_rows, index=ua_cols).T,
    "RH (%)":                  ua_table("r",  lambda x: x,          1),
    "Sp humidity (g/kg)":      ua_table("q",  lambda x: x * 1000,   3),
    "Vorticity (×10⁻⁵ s⁻¹)":  ua_table("vo", lambda x: x * 1e5,    2),
    "Divergence (×10⁻⁵ s⁻¹)": ua_table("d",  lambda x: x * 1e5,    2),
    **( {"Vert velocity (Pa/s)": ua_table("w", lambda x: x, 3)} if "w" in ua_vars else {} ),
}

print()
print("UPPER AIR  (12-hourly steps)")
for title, table in ua_tables.items():
    print()
    print(f"  {title}")
    print("  " + "-" * 50)
    print(table.to_string(col_space=10).replace("\n", "\n  "))

In [ ]:
import sys
from datetime import datetime, timezone

output_path = RUN_DIR / f"analysis_{RUN_DIR.name}.txt"

class Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, text):
        for s in self.streams: s.write(text)
    def flush(self):
        for s in self.streams: s.flush()

# ── Text-output helpers ───────────────────────────────────────────────────────
def txt_label(index):
    """Single-line step label: 'T+NNN  DDMon HH:MM (AST)'"""
    labels = []
    for ts in pd.to_datetime(index):
        step_h = int((ts - run_init).total_seconds() / 3600)
        lt     = (ts + UTC_OFFSET).strftime("%d%b %H:%M")
        labels.append(f"T+{step_h:03d}  {lt}")
    return labels

# Abbreviated column names for the transposed surface / wave tables
SFC_SHORT = {
    "T2m (°C)":       "T2m°C",
    "Dewpoint (°C)":  "Dewp°C",
    "MSL (hPa)":      "MSL",
    "SP (hPa)":       "SP",
    "Wind dir":       "Dir",
    "Wind (kt)":      "Wind",
    "Gust (kt)":      "Gust",
    "Cloud (%)":      "Cld%",
    "Precip 3h (mm)": "Prcp",
    "Runoff 3h (mm)": "Rnff",
    "TCWV (kg/m²)":   "TCWV",
    "MUCAPE (J/kg)":  "MUCAPE",
}
WAV_SHORT = {
    "SWH (m)":         "SWH m",
    "SWH (ft)":        "SWH ft",
    "Dir":             "Dir",
    "Mean period (s)": "MeanPd",
    "Peak period (s)": "PeakPd",
}

# float_format per upper-air table title
UA_FMT = {
    "Temperature (°C)":        "%.1f",
    "Geopotential ht (m)":     "%.0f",
    "Wind speed (kt)":         "%.1f",
    "RH (%)":                  "%.1f",
    "Sp humidity (g/kg)":      "%.3f",
    "Vorticity (×10⁻⁵ s⁻¹)":  "%.2f",
    "Divergence (×10⁻⁵ s⁻¹)": "%.2f",
    "Vert velocity (Pa/s)":    "%.3f",
}

with open(output_path, "w", encoding="utf-8") as f:
    tee = Tee(sys.stdout, f)

    def p(*args, **kwargs):
        print(*args, **{**kwargs, "file": tee})

    # ── Header ────────────────────────────────────────────────────────────────
    p("=" * 80)
    p("ECMWF AIFS-SINGLE ANALYSIS — TGPY / GRENADA")
    p("Maurice Bishop International Airport  |  12.0000°N, 61.7500°W")
    p(f"Run       : {RUN_DIR.name}  (all times AST = UTC−4)")
    p(f"Generated : {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}")
    p(f"Grid point: {actual_lat:.4f}°N, {actual_lon:.4f}°W  (~{dist_km:.1f} km from TGPY)")
    p("=" * 80)

    # ── T+24h snapshot ────────────────────────────────────────────────────────
    snap_vt  = str(sfc["msl"].valid_time.values)[:19].replace("T", " ")
    snap_ast = (pd.Timestamp(str(sfc["msl"].valid_time.values)[:19]) + UTC_OFFSET).strftime("%d %b %H:%M AST")
    p(f"\n{'─' * 80}")
    p(f"  SURFACE SNAPSHOT — T+24h  (valid {snap_ast} / {snap_vt} UTC)")
    p(f"{'─' * 80}")
    p(f"  2m Temperature : {t2m_c:.1f} °C")
    p(f"  MSL Pressure   : {msl_hpa:.1f} hPa")
    p(f"  10m Wind       : {wspd * 1.944:.1f} kt  {deg_to_cardinal(wdir)}  ({wdir:.0f}°)")

    ua_vt_snap = str(ua.valid_time.values)[:19].replace("T", " ")
    p(f"\n  UPPER-AIR PROFILE — T+24h  (valid {ua_vt_snap} UTC)")
    p(f"  {'Level':>6}  {'Temp':>6}  {'GH':>6}  {'Wind':>8}  {'Dir':>6}")
    p(f"  {'(hPa)':>6}  {'(°C)':>6}  {'(m)':>6}  {'(kt)':>8}  {'':>6}")
    p(f"  {'─'*6}  {'─'*6}  {'─'*6}  {'─'*8}  {'─'*6}")
    for lev in levels:
        snap_pt = ua.sel(isobaricInhPa=lev)
        t_c  = float(snap_pt["t"]) - 273.15
        gh   = float(snap_pt["gh"])
        u    = float(snap_pt["u"])
        v    = float(snap_pt["v"])
        spd  = np.sqrt(u**2 + v**2) * 1.944
        wd   = (270 - np.degrees(np.arctan2(v, u))) % 360
        p(f"  {lev:>6.0f}  {t_c:>6.1f}  {gh:>6.0f}  {spd:>8.1f}  {deg_to_cardinal(wd):>6}")

    # ── Section 5: 6-hourly surface time series ───────────────────────────────
    p(f"\n{'─' * 80}")
    p(f"  SURFACE TIME SERIES — 6-hourly  (T+0 to T+120h)")
    p(f"{'─' * 80}")
    p(f"  {'Step':>5}  {'AST':>14}  {'T2m°C':>6}  {'MSL hPa':>8}  {'Wind kt':>8}  {'Dir':>5}")
    p(f"  {'─'*5}  {'─'*14}  {'─'*6}  {'─'*8}  {'─'*8}  {'─'*5}")
    for r in rows:
        ast = (pd.Timestamp(r["valid_time"]) + UTC_OFFSET).strftime("%d%b %H:%M")
        p(f"  {r['step']:>5}  {ast:>14}  {r['t2m_c']:>6.1f}  {r['msl_hpa']:>8.1f}  {r['wspd_kt']:>8.1f}  {deg_to_cardinal(r['wdir']):>5}")

    # ── Section 6: 3-hourly surface — rows = time steps, cols = variables ─────
    p(f"\n{'─' * 80}")
    p(f"  SURFACE — 3-hourly  (T+0 to T+120h)")
    p(f"  Columns: T2m°C Dewp°C MSL(hPa) SP(hPa) Dir Wind(kt) Gust(kt) Cld% Prcp(mm) Rnff(mm) TCWV(kg/m²) MUCAPE(J/kg)")
    p(f"{'─' * 80}")
    sfc_out         = sfc_table.T.copy()
    sfc_out.index   = txt_label(df_sfc.index)
    sfc_out.columns = [SFC_SHORT.get(c, c) for c in sfc_out.columns]
    p(sfc_out.to_string())

    # ── Section 6: 3-hourly wave — rows = time steps, cols = variables ────────
    p(f"\n{'─' * 80}")
    p(f"  WAVE — 3-hourly  (T+0 to T+120h)")
    p(f"  Columns: SWH(m) SWH(ft) Dir MeanPeriod(s) PeakPeriod(s)")
    p(f"{'─' * 80}")
    wav_out         = wav_table.T.copy()
    wav_out.index   = txt_label(df_wave.index)
    wav_out.columns = [WAV_SHORT.get(c, c) for c in wav_out.columns]
    p(wav_out.to_string())

    # ── Section 6: upper air — levels as rows, steps as columns ───────────────
    p(f"\n{'─' * 80}")
    p(f"  UPPER AIR — 12-hourly  (T+0 to T+120h)")
    p(f"{'─' * 80}")
    ua_cols_txt = [c.replace("\n", " ") for c in ua_cols]
    for title, table in ua_tables.items():
        p(f"\n  {title}")
        p("  " + "─" * 60)
        tbl         = table.copy()
        tbl.columns = ua_cols_txt
        fmt_str     = UA_FMT.get(title, "%.1f")
        fmt_fn      = lambda x, f=fmt_str: f % x if isinstance(x, (int, float)) else str(x)
        p(tbl.to_string(col_space=14, float_format=fmt_fn).replace("\n", "\n  "))

    p("\n" + "=" * 80)
    p("END OF REPORT")
    p("=" * 80)

print(f"\nSaved: {output_path.resolve()}")

In [ ]:
# ── Canonical CSV export — contract for ensemble/compare.ipynb heatmap ────────
CANONICAL_COLS = [
    # Core (original 11)
    "t2m_c", "msl_hpa", "wspd_kt", "wdir", "d2m_c", "tcc_pct",
    "tp_mm", "tcwv", "fg10_kt", "sp_hpa", "cape",
    # New tropical params (AIFS-specific gaps: olr_wm2, blh_m, cin, lhf, shf are NaN)
    "skt_c", "cp_mm", "lcc_pct", "mcc_pct", "hcc_pct",
    "cin", "olr_wm2", "blh_m", "lhf", "shf",
]

base = df_sfc.iloc[::2].copy()   # 3-hourly → 6-hourly

export = pd.DataFrame(
    {col: base[col].values if col in base.columns else [float("nan")] * len(base)
     for col in CANONICAL_COLS},
    index=base.index,
)
export.insert(0, "model", "ecmwf_aifs")
export.index.name = "valid_time"

csv_path = RUN_DIR / "canonical_sfc.csv"
export.to_csv(csv_path, float_format="%.4f")
non_nan = export.notna().sum().sum()
print(f"Saved: {csv_path.name}  ({len(export)} steps, {non_nan} non-NaN values)")
print(export[["t2m_c", "msl_hpa", "wspd_kt", "skt_c", "cp_mm", "lcc_pct"]].round(2).to_string())